# Importing GDACS Disaster API

Main goal of this file : Collect real-time disaster alerts from GDACS.

Fields to extract:

- Disaster Type
- Event Name
- Severity (if available)
- Latitude
- Longitude
- Published Date

Output:

gdacs_data.csv

In [1]:
import feedparser
import pandas as pd
import requests
import xml.etree.ElementTree as ET

RSS_URL = "https://www.gdacs.org/xml/rss.xml"

feed = feedparser.parse(RSS_URL)

print("Total Events :", len(feed.entries))

Total Events : 213


In [2]:
feed.entries[0]

{'title': 'Green earthquake (Magnitude 5.5M, Depth:10km) in Southern East Pacific Rise 12/07/2026 07:29 UTC, [unknown].',
 'title_detail': {'type': 'text/plain',
  'language': None,
  'base': 'https://www.gdacs.org/xml/rss.xml',
  'value': 'Green earthquake (Magnitude 5.5M, Depth:10km) in Southern East Pacific Rise 12/07/2026 07:29 UTC, [unknown].'},
 'summary': 'On 7/12/2026 7:29:23 AM, an earthquake occurred in Southern East Pacific Rise potentially affecting [unknown]. The earthquake had Magnitude 5.5M, Depth:10km.',
 'summary_detail': {'type': 'text/html',
  'language': None,
  'base': 'https://www.gdacs.org/xml/rss.xml',
  'value': 'On 7/12/2026 7:29:23 AM, an earthquake occurred in Southern East Pacific Rise potentially affecting [unknown]. The earthquake had Magnitude 5.5M, Depth:10km.'},
 'links': [{'type': 'image/png',
   'length': '1',
   'href': 'https://www.gdacs.org/contentdata/resources/imgtemp/gdacs/eq/eq1717695_1.png',
   'rel': 'enclosure'},
  {'rel': 'alternate',
   '

In [3]:
records = []

for event in feed.entries:
    records.append({
        "GDACSEventID": event.get("gdacs_eventid"),
        "DisasterType": event.get("gdacs_eventtype"),
        "AlertLevel": event.get("gdacs_alertlevel"),
        "Severity": event.get("gdacs_severity"),
        "Country": event.get("gdacs_country"),
        "Latitude": event.get("geo_lat"),
        "Longitude": event.get("geo_long"),
        "Title": event.get("title"),
        "Published": event.get("published"),
        "Link": event.get("link"),
        "Summary": event.get("summary"),
    })

gdacs_df = pd.DataFrame(records)

gdacs_df.head()

,GDACSEventID,DisasterType,AlertLevel,Severity,Country,Latitude,Longitude,Title,Published,Link,Summary
0,1551421,EQ,Green,"{'unit': 'M', 'value': '5.5'}",Southern East Pacific Rise,-54.6008,-118.9732,"Green earthquake (Magnitude 5.5M, Depth:10km) ...","Sun, 12 Jul 2026 07:47:57 GMT",https://www.gdacs.org/report.aspx?eventtype=EQ...,"On 7/12/2026 7:29:23 AM, an earthquake occurre..."
1,1551279,EQ,Green,"{'unit': 'M', 'value': '6.4'}",South Sandwich Islands Region,-55.5424,-28.8526,"Green earthquake (Magnitude 6.4M, Depth:26km) ...","Sat, 11 Jul 2026 10:45:18 GMT",https://www.gdacs.org/report.aspx?eventtype=EQ...,"On 7/11/2026 10:26:45 AM, an earthquake occurr..."
2,1000143,VO,Green,"{'unit': '', 'value': '0'}",Philippines,10.412,123.132,Volcanic eruption is on going for Kanlaon in P...,"Fri, 10 Jul 2026 06:03:01 GMT",https://www.gdacs.org/report.aspx?eventtype=VO...,Volcano Kanlaon is emitting ash clouds accordi...
3,1029474,WF,Green,"{'unit': 'ha', 'value': '5143'}",Australia,-16.527741074717682,145.0224129171192,Green forest fire notification in Australia,"Sun, 12 Jul 2026 15:00:42 GMT",https://www.gdacs.org/report.aspx?eventtype=WF...,"On 10/07/2026, a forest fire started in Austra..."
4,1104013,FL,Green,"{'unit': '', 'value': '0'}",Guatemala,15.3232265,-89.6686416,Green flood alert in Guatemala,"Sat, 11 Jul 2026 06:40:35 GMT",https://www.gdacs.org/report.aspx?eventtype=FL...,"On 09/07/2026, a flood started in Guatemala, l..."


In [4]:
gdacs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 213 entries, 0 to 212
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   GDACSEventID  213 non-null    object
 1   DisasterType  213 non-null    object
 2   AlertLevel    213 non-null    object
 3   Severity      213 non-null    object
 4   Country       213 non-null    object
 5   Latitude      213 non-null    object
 6   Longitude     213 non-null    object
 7   Title         213 non-null    object
 8   Published     213 non-null    object
 9   Link          213 non-null    object
 10  Summary       213 non-null    object
dtypes: object(11)
memory usage: 18.4+ KB


In [5]:
gdacs_df.isnull().sum()

GDACSEventID    0
DisasterType    0
AlertLevel      0
Severity        0
Country         0
Latitude        0
Longitude       0
Title           0
Published       0
Link            0
Summary         0
dtype: int64

In [6]:
gdacs_df.to_csv("../datasets/gdacs_data.csv", index=False)

print("GDACS Dataset Saved")

GDACS Dataset Saved


In [7]:
import os

os.listdir("../datasets")

['combined_disasters.csv',
 'earthquake_data.csv',
 'event_infrastructure_summary.csv',
 'final_disaster_dataset.csv',
 'final_disaster_dataset_engineered.csv',
 'final_disaster_dataset_with_infrastructure.csv',
 'gdacs_data.csv',
 'india-latest.osm.pbf',
 'india_infrastructure.parquet',
 'osm_checkpoints',
 'osm_regions',
 'population_clean.npy',
 'population_data.tif',
 'population_metadata.json',
 'population_selected_metadata.json',
 'shapefiles',
 'useful_disaster_events.csv',
 'weather_data.csv',
 'weather_useful_events.csv']

In [8]:
#The RSS feed doesn't always include separate latitude, longitude, severity, and disaster type as simple fields.

#For this project, it's better to use the GDACS GeoJSON feed, which provides richer structured data.

url = "https://www.gdacs.org/gdacsapi/api/events/geteventlist/SEARCH"

response = requests.get(url)

print(response.status_code)

200


In [9]:
gdacs_df.head(5)

,GDACSEventID,DisasterType,AlertLevel,Severity,Country,Latitude,Longitude,Title,Published,Link,Summary
0,1551421,EQ,Green,"{'unit': 'M', 'value': '5.5'}",Southern East Pacific Rise,-54.6008,-118.9732,"Green earthquake (Magnitude 5.5M, Depth:10km) ...","Sun, 12 Jul 2026 07:47:57 GMT",https://www.gdacs.org/report.aspx?eventtype=EQ...,"On 7/12/2026 7:29:23 AM, an earthquake occurre..."
1,1551279,EQ,Green,"{'unit': 'M', 'value': '6.4'}",South Sandwich Islands Region,-55.5424,-28.8526,"Green earthquake (Magnitude 6.4M, Depth:26km) ...","Sat, 11 Jul 2026 10:45:18 GMT",https://www.gdacs.org/report.aspx?eventtype=EQ...,"On 7/11/2026 10:26:45 AM, an earthquake occurr..."
2,1000143,VO,Green,"{'unit': '', 'value': '0'}",Philippines,10.412,123.132,Volcanic eruption is on going for Kanlaon in P...,"Fri, 10 Jul 2026 06:03:01 GMT",https://www.gdacs.org/report.aspx?eventtype=VO...,Volcano Kanlaon is emitting ash clouds accordi...
3,1029474,WF,Green,"{'unit': 'ha', 'value': '5143'}",Australia,-16.527741074717682,145.0224129171192,Green forest fire notification in Australia,"Sun, 12 Jul 2026 15:00:42 GMT",https://www.gdacs.org/report.aspx?eventtype=WF...,"On 10/07/2026, a forest fire started in Austra..."
4,1104013,FL,Green,"{'unit': '', 'value': '0'}",Guatemala,15.3232265,-89.6686416,Green flood alert in Guatemala,"Sat, 11 Jul 2026 06:40:35 GMT",https://www.gdacs.org/report.aspx?eventtype=FL...,"On 09/07/2026, a flood started in Guatemala, l..."
